# Group Work 3：線形代数による乳がん分類（教員用解答例）

`gw3_ex.ipynb`に対応する教員用資料である．受講生には課題提出後に配布することを想定している．数値解だけでなく，shapeの検証，損失の推移，未知データでの評価を含む．

## 0．準備

次のセルは変更しない．`wdbc_data`ディレクトリから，訓練データ300件と検証データ269件を読み込む．

In [1]:
from pathlib import Path
import numpy as np

np.set_printoptions(precision=5, suppress=True)
DATA_DIR = Path("wdbc_data")

def load_split(name, n_rows):
    x = np.loadtxt(DATA_DIR / f"{name}_A.data", dtype=float).reshape(n_rows, 30)
    y = np.loadtxt(DATA_DIR / f"{name}_b.data", dtype=float).reshape(n_rows, 1)
    return x, y

x_train, y_train = load_split("train", 300)
x_valid, y_valid = load_split("validate", 269)
print("train:", x_train.shape, y_train.shape)
print("valid:", x_valid.shape, y_valid.shape)

FEATURE_NAMES = [
    "radius", "texture", "perimeter", "area", "smoothness",
    "compactness", "concavity", "concave_pts", "symmetry", "fractal_dim",
]


def print_w(w):
    """30次元の係数を，特徴量ごとにmean，StdErr，worstの順で表示する．"""
    values = np.asarray(w, dtype=float).reshape(-1)
    if values.size != 30:
        raise ValueError("w must contain 30 elements")

    print(f'{"feature":<14}{"mean":>13}{"StdErr":>13}{"worst":>13}')
    print("-" * 53)
    for i, name in enumerate(FEATURE_NAMES):
        print(f"{name:<14}{values[i]:13.6f}{values[i + 10]:13.6f}{values[i + 20]:13.6f}")


def _classification_counts(mA, vb, vw):
    actual = np.asarray(vb).reshape(-1)
    scores = (np.asarray(mA) @ np.asarray(vw)).reshape(-1)
    predicted = np.where(scores >= 0, 1, -1)

    tp = int(np.sum((actual == 1) & (predicted == 1)))
    tn = int(np.sum((actual == -1) & (predicted == -1)))
    fp = int(np.sum((actual == -1) & (predicted == 1)))
    fn = int(np.sum((actual == 1) & (predicted == -1)))
    total = actual.size
    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "correct": tp + tn,
        "total": total,
        "accuracy": (tp + tn) / total,
    }


def show_accuracy(mA, vb, vw):
    """正解数，accuracy，偽陽性，偽陰性を簡潔に表示する．"""
    result = _classification_counts(mA, vb, vw)
    print(f"correct       : {result['correct']:4d} / {result['total']:4d}")
    print(f"accuracy      : {result['accuracy']:8.3%}")
    print(f"false positive: {result['fp']:4d}  (良性を悪性と予測)")
    print(f"false negative: {result['fn']:4d}  (悪性を良性と予測)")
    return result


def show_accuracy2(mA, vb, vw):
    """混同行列を表示する．positiveは悪性（+1）である．"""
    result = _classification_counts(mA, vb, vw)
    print(" " * 22 + "predicted")
    print(f'{"":22}{"malignant(+1)":>15}{"benign(-1)":>13}')
    print(f'{"actual malignant(+1)":22}{result["tp"]:15d}{result["fn"]:13d}')
    print(f'{"actual benign(-1)":22}{result["fp"]:15d}{result["tn"]:13d}')
    print(f"accuracy: {result['accuracy']:.3%}")
    return result

train: (300, 30) (300, 1)
valid: (269, 30) (269, 1)


## 1．データとshape

1．`x_train`，`y_train`のshapeを記せ．各軸が何を表すか説明せよ．  
2．ラベルに含まれる値と，それぞれの件数を求めよ．  
3．30要素すべてが`0.0001`である列ベクトル`w0`を作り，`x_train @ w0`のshapeを確かめよ．

In [2]:
print("x_train:", x_train.shape, "（標本数，特徴量数）")
print("y_train:", y_train.shape, "（標本数，1）")

labels, counts = np.unique(y_train, return_counts=True)
print("labels:", dict(zip(labels.astype(int), counts)))

w0 = np.full((x_train.shape[1], 1), 0.0001)
initial_scores = x_train @ w0
print("w0:", w0.shape)
print("scores:", initial_scores.shape)

x_train: (300, 30) （標本数，特徴量数）
y_train: (300, 1) （標本数，1）
labels: {-1: 154, 1: 146}
w0: (30, 1)
scores: (300, 1)


**説明：**  
`x_train`は300標本×30特徴量，`y_train`は300標本×1列である．$A(300,30)w(30,1)$により，1標本につき1個のスコアを持つ$(300,1)$の列ベクトルが得られる．ラベル集計は悪性$+1$が146件，良性$-1$が154件である．

## 2．スコアと損失

損失を

$$
J(w)=\frac{1}{2}\lVert Aw-b\rVert_2^2
$$

とする．次を行え．

1．任意の`w`に対して損失を返す関数`squared_loss(A, b, w)`を実装せよ．  
2．`w0`に対する訓練データの損失を求めよ．  
3．最初の10件について，スコア，予測ラベル，正解ラベルを表示せよ．スコアが0以上なら$+1$，0未満なら$-1$とする．

In [3]:
def squared_loss(A, b, w):
    residual = A @ w - b
    return 0.5 * float(np.linalg.norm(residual) ** 2)

initial_loss = squared_loss(x_train, y_train, w0)
initial_predictions = np.where(initial_scores >= 0, 1, -1)
print("initial loss:", initial_loss)
print(" score     pred  true")
for score, pred, true in zip(initial_scores[:10, 0], initial_predictions[:10, 0], y_train[:10, 0]):
    print(f"{score:8.4f} {pred:6d} {int(true):5d}")

initial loss: 137.24583380757323
 score     pred  true
  0.3566      1     1
  0.3741      1     1
  0.3387      1     1
  0.1240      1     1
  0.3337      1     1
  0.1506      1     1
  0.3068      1     1
  0.1814      1     1
  0.1565      1     1
  0.1493      1     1


**考察：**  
損失はスコアと$\pm1$の正解ラベルとの距離を測るが，分類はスコアの符号だけで決まる．例えば正解が$+1$なら，スコア0.1と0.9はどちらも正解だが損失は異なる．したがって，損失の減少と正解数の増加は同じ意味ではない．

## 3．勾配と1回の更新

$$
\nabla J(w)=A^\mathsf{T}(Aw-b)
$$

を用いる．

1．任意の`w`に対して勾配を返す関数`loss_gradient(A, b, w)`を実装せよ．  
2．`w0`で勾配を求め，そのshapeが`w0`と等しいことを確認せよ．  
3．学習率`eta = 3.0e-9`で1回更新した`w1`を求め，更新前後の損失を比較せよ．

In [4]:
def loss_gradient(A, b, w):
    return A.T @ (A @ w - b)

eta = 3.0e-9
gradient0 = loss_gradient(x_train, y_train, w0)
w1 = w0 - eta * gradient0
print("gradient:", gradient0.shape, "w0:", w0.shape)
print("before:", squared_loss(x_train, y_train, w0))
print("after :", squared_loss(x_train, y_train, w1))

gradient: (30, 1) w0: (30, 1)
before: 137.24583380757323
after : 135.73834994042608


**説明：**  
$A^\mathsf{T}$は$(30,300)$，残差$Aw-b$は$(300,1)$なので，積は$(30,1)$となる．これは更新対象の$w$と同じshapeである．

## 4．勾配降下法

`w0`から出発し，300回更新したベクトル`w_gd`を求めよ．各反復の更新後の損失をリスト`loss_history`に保存すること．

さらに，次を報告せよ．

- 初期損失と最終損失
- 1，10，50，100，300回目の損失
- `print_w(w_gd)`による係数一覧
- `show_accuracy(x_train, y_train, w_gd)`による訓練結果

`loss_history`が300要素であること，すべて有限値であることも確認せよ．

In [5]:
def accuracy(A, b, w):
    predictions = np.where(A @ w >= 0, 1, -1)
    return float(np.mean(predictions == b))

n_steps = 300
w_gd = w0.copy()
loss_history = []

for _ in range(n_steps):
    w_gd = w_gd - eta * loss_gradient(x_train, y_train, w_gd)
    loss_history.append(squared_loss(x_train, y_train, w_gd))

print("initial loss:", initial_loss)
for step in (1, 10, 50, 100, 300):
    print(f"step {step:3d}: {loss_history[step - 1]:.6f}")
print("history length:", len(loss_history))
print("all finite:", np.isfinite(loss_history).all())
print("\n--- w_gd ---")
print_w(w_gd)
print("\n--- training summary ---")
show_accuracy(x_train, y_train, w_gd)

initial loss: 137.24583380757323
step   1: 135.738350
step  10: 130.648515
step  50: 118.236799
step 100: 111.135183
step 300: 101.499258
history length: 300
all finite: True

--- w_gd ---
feature                mean       StdErr        worst
-----------------------------------------------------
radius            -0.000334     0.000097    -0.000319
texture           -0.000655     0.000029    -0.000839
perimeter         -0.002493     0.000097    -0.002343
area              -0.001718     0.000971     0.001933
smoothness         0.000095     0.000100     0.000094
compactness        0.000100     0.000100     0.000104
concavity          0.000104     0.000099     0.000108
concave_pts        0.000102     0.000100     0.000102
symmetry           0.000091     0.000099     0.000088
fractal_dim        0.000096     0.000100     0.000096

--- training summary ---
correct       :  274 /  300
accuracy      :  91.333%
false positive:    5  (良性を悪性と予測)
false negative:   21  (悪性を良性と予測)


{'tp': 125,
 'tn': 149,
 'fp': 5,
 'fn': 21,
 'correct': 274,
 'total': 300,
 'accuracy': 0.9133333333333333}

**考察：**  
損失は初期値から大きく低下し，後半ほど変化が緩やかになるため，最小値へ近づいていると判断できる．学習率を10倍にすると，このデータでは更新量が大きすぎて損失が振動・発散する可能性が高い．学習率の適否は損失履歴で確認すべきである．

## 5．QR分解による最小二乗解

`np.linalg.qr`のreduced QR分解を用い，`x_train = Q @ R`とせよ．

1．`Q`と`R`のshapeを確認せよ．  
2．`Q.T @ y_train`を計算し，そのshapeと最初の5要素を表示せよ．  
3．`w_qr = np.linalg.solve(R, Q.T @ y_train)`として$Rw=Q^\mathsf{T}b$を解け．逆行列を明示的に作らないこと．  
4．`predicted_b = x_train @ w_qr`を求め，正解`y_train`と予測値`predicted_b`を最初の10件について横に並べて表示せよ．  
5．`w_qr`の損失を求め，`print_w`と`show_accuracy`で結果を表示し，勾配降下法と比較せよ．

In [6]:
Q, R = np.linalg.qr(x_train, mode="reduced")
qt_b = Q.T @ y_train
w_qr = np.linalg.solve(R, Q.T @ y_train)
predicted_b = x_train @ w_qr

print("Q:", Q.shape, "R:", R.shape, "Q.T @ y_train:", qt_b.shape)
print("first five elements of Q.T @ y_train:\n", qt_b[:5])
print("\n       b  predicted b")
for actual, predicted in zip(y_train[:10, 0], predicted_b[:10, 0]):
    print(f"{actual:8.3f}{predicted:13.3f}")

print("QR loss:", squared_loss(x_train, y_train, w_qr))
print("GD loss:", squared_loss(x_train, y_train, w_gd))
print("\n--- w_qr ---")
print_w(w_qr)
print("\n--- QR training summary ---")
show_accuracy(x_train, y_train, w_qr)

Q: (300, 30) R: (30, 30) Q.T @ y_train: (30, 1)
first five elements of Q.T @ y_train:
 [[-2.44163]
 [-4.07999]
 [-8.84787]
 [ 6.36606]
 [-0.81285]]

       b  predicted b
   1.000        0.967
   1.000        0.518
   1.000        1.354
   1.000        1.208
   1.000        0.679
   1.000        0.430
   1.000        0.802
   1.000        0.266
   1.000        0.586
   1.000        1.467
QR loss: 38.75025548203763
GD loss: 101.49925832671087

--- w_qr ---
feature                mean       StdErr        worst
-----------------------------------------------------
radius            -0.869922     0.956592     0.408606
texture            0.024314     0.082053     0.003346
perimeter          0.062680     0.007943     0.000678
area               0.003275    -0.004977    -0.002511
smoothness         8.790305    27.841938    -4.531369
compactness       -1.747148    -3.301519    -0.590111
concavity          0.202848    -4.985957     0.719369
concave_pts        6.506449    16.318887     2.158967


{'tp': 133,
 'tn': 153,
 'fp': 1,
 'fn': 13,
 'correct': 286,
 'total': 300,
 'accuracy': 0.9533333333333334}

**考察：**  
QR分解は最小二乗問題を三角連立方程式に変換して直接解くため，300回で打ち切った勾配降下法より小さい損失を得る．勾配降下法は学習率と反復回数に依存し，300回では最小二乗解へ十分に収束していない．一方，大規模問題では反復法がメモリや計算量の点で有利になる場合もある．

## 6．未知データでの評価

準備セルで定義済みの`show_accuracy`と`show_accuracy2`を用いる．悪性（$+1$）をpositiveとする．

1．4項目の合計が標本数になることを確認せよ．  
2．`w_gd`と`w_qr`を，訓練データと検証データの両方で評価せよ．要約には`show_accuracy`，混同行列には`show_accuracy2`を使うこと．  
3．どちらのモデルを採用するか，損失，accuracy，FN，FPを根拠に述べよ．

In [7]:
models = {"gradient descent": w_gd, "QR": w_qr}
datasets = {"train": (x_train, y_train), "validation": (x_valid, y_valid)}

for model_name, weights in models.items():
    for split_name, (features, labels) in datasets.items():
        report = _classification_counts(features, labels, weights)
        assert report["tp"] + report["tn"] + report["fp"] + report["fn"] == labels.size
        loss = squared_loss(features, labels, weights)
        print(f"\n=== {model_name} / {split_name} ===")
        print(f"loss: {loss:.4f}")
        show_accuracy(features, labels, weights)
        show_accuracy2(features, labels, weights)


=== gradient descent / train ===
loss: 101.4993
correct       :  274 /  300
accuracy      :  91.333%
false positive:    5  (良性を悪性と予測)
false negative:   21  (悪性を良性と予測)
                      predicted
                        malignant(+1)   benign(-1)
actual malignant(+1)              125           21
actual benign(-1)                   5          149
accuracy: 91.333%

=== gradient descent / validation ===
loss: 99.0249
correct       :  248 /  269
accuracy      :  92.193%
false positive:   10  (良性を悪性と予測)
false negative:   11  (悪性を良性と予測)
                      predicted
                        malignant(+1)   benign(-1)
actual malignant(+1)               55           11
actual benign(-1)                  10          193
accuracy: 92.193%

=== QR / train ===
loss: 38.7503
correct       :  286 /  300
accuracy      :  95.333%
false positive:    1  (良性を悪性と予測)
false negative:   13  (悪性を良性と予測)
                      predicted
                        malignant(+1)   benign(-1)
actual malignant(+

**最終考察：**  
QR解を採用する．訓練損失は勾配降下法の101.50に対して38.75と小さく，検証accuracyも92.2%から97.0%へ向上した．さらに，検証データのFNは11件から2件，FPは10件から6件に減った．特にFNは悪性の見落としなので重く評価する．ただし，各特徴量は面積や滑らかさなど尺度が異なるため，標準化していない生の係数の絶対値を，そのまま特徴の重要度とは解釈できない．

**外部ツール等の利用申告：**  
解答例のため省略．